In [ ]:
from dotenv import load_dotenv
load_dotenv()

import re, math
from collections import Counter
import voyageai

vo_client = voyageai.Client()

def chunk_by_section(text):
    return re.split(r"\n## ", text)

def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    inp = chunks if is_list else [chunks]
    result = vo_client.embed(inp, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [ ]:
# VectorIndex + BM25Index (from previous lessons)
class VectorIndex:
    def __init__(self, embedding_fn=None):
        self.vectors, self.documents = [], []
        self._vector_dim, self._embedding_fn = None, embedding_fn
    def add_documents(self, docs):
        vectors = self._embedding_fn([d["content"] for d in docs])
        for v, d in zip(vectors, docs): self.add_vector(v, d)
    def add_vector(self, vector, doc):
        if not self.vectors: self._vector_dim = len(vector)
        self.vectors.append(list(vector)); self.documents.append(doc)
    def search(self, query, k=1):
        qv = self._embedding_fn(query) if isinstance(query, str) else query
        dists = []
        for i, s in enumerate(self.vectors):
            dot = sum(a*b for a,b in zip(qv,s))
            m1, m2 = math.sqrt(sum(x*x for x in qv)), math.sqrt(sum(x*x for x in s))
            dists.append((1.0 - dot/(m1*m2) if m1 and m2 else 1.0, self.documents[i]))
        dists.sort(key=lambda x: x[0])
        return [(d, dist) for dist, d in dists[:k]]

class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.documents, self._corpus_tokens, self._doc_len = [], [], []
        self._doc_freqs, self._idf = {}, {}
        self._index_built, self.k1, self.b = False, k1, b
    def _tokenize(self, text): return [t for t in re.split(r"\W+", text.lower()) if t]
    def add_documents(self, docs):
        for d in docs:
            tokens = self._tokenize(d["content"])
            self.documents.append(d); self._corpus_tokens.append(tokens); self._doc_len.append(len(tokens))
            seen = set()
            for t in tokens:
                if t not in seen: self._doc_freqs[t] = self._doc_freqs.get(t,0)+1; seen.add(t)
        self._index_built = False
    def search(self, query, k=1, nf=0.1):
        if not self._index_built:
            N = len(self.documents)
            self._avg_dl = sum(self._doc_len)/N if N else 0
            self._idf = {t: math.log(((N-f+0.5)/(f+0.5))+1) for t,f in self._doc_freqs.items()}
            self._index_built = True
        tokens = self._tokenize(query)
        raw = []
        for i in range(len(self.documents)):
            s = 0.0; counts = Counter(self._corpus_tokens[i]); dl = self._doc_len[i]
            for t in tokens:
                if t in self._idf:
                    tf = counts.get(t,0)
                    s += (self._idf[t]*tf*(self.k1+1))/(tf+self.k1*(1-self.b+self.b*(dl/self._avg_dl))+1e-9)
            if s > 1e-9: raw.append((s, self.documents[i]))
        raw.sort(key=lambda x: x[0], reverse=True)
        return [(d, math.exp(-nf*s)) for s,d in raw[:k]]

In [ ]:
# Retriever — hybrid search with Reciprocal Rank Fusion
class Retriever:
    def __init__(self, *indexes):
        self._indexes = list(indexes)
    def add_documents(self, docs):
        for idx in self._indexes: idx.add_documents(docs)
    def search(self, query, k=1, k_rrf=60):
        all_results = [idx.search(query, k=k*5) for idx in self._indexes]
        doc_ranks = {}
        for i, results in enumerate(all_results):
            for rank, (doc, _) in enumerate(results):
                did = id(doc)
                if did not in doc_ranks:
                    doc_ranks[did] = {"doc": doc, "ranks": [float("inf")] * len(self._indexes)}
                doc_ranks[did]["ranks"][i] = rank + 1
        scored = [(info["doc"], sum(1.0/(k_rrf+r) for r in info["ranks"] if r != float("inf")))
                  for info in doc_ranks.values()]
        scored = [(d,s) for d,s in scored if s > 0]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]

In [ ]:
# Build hybrid index
with open("report.md", "r") as f:
    text = f.read()
chunks = chunk_by_section(text)

retriever = Retriever(BM25Index(), VectorIndex(embedding_fn=generate_embedding))
retriever.add_documents([{"content": c} for c in chunks])
print(f"{len(chunks)} chunks indexed")

In [ ]:
# Search (run one at a time to avoid rate limits)
query = "What happened with INC-2023-Q4-011?"
for doc, score in retriever.search(query, k=3):
    print(f"  RRF={score:.4f}: {doc['content'].strip().split(chr(10))[0][:80]}")